# combine

> Combine calibrated dithers into a single mosaic using swarp.

In [ ]:
# | default_exp euclid.combine

In [ ]:
# | exporti

import tempfile
from pathlib import Path
from abc import ABC, abstractmethod
import subprocess
import socket
import getpass
from itertools import chain
from shutil import copy2


import numpy as np
from astropy.io import fits
from astropy.stats import SigmaClip
from astropy import units as u
from astropy.coordinates import SkyCoord
from photutils.background import Background2D

from nicl.euclid.data_access import DataAccess
from nicl.mask import fast_mask
from nicl.euclid.constants import (
    VIS,
    NISP,
    MER,
    SWARP_CONFIG_NISP,
    SWARP_CONFIG_VIS,
    SWARP_CONFIG_MER,
)
from nicl.euclid.utilities import (
    default_data_path,
    round_up_box_size,
    get_dither_id_from_filename,
)

In [ ]:
# | hide
# additional imports for examples

from nicl.utilities import physical_to_angular

In [ ]:
# | export
# base class for combining images


class Combiner(ABC):
    """Combines Euclid images using SWarp"""

    def __init__(
        self,
        in_dir=None,  # directory at which to find the input images
        out_dir=None,  # directory at which to place output images
        filters=None,  # list of filters to combine
        ids=None,  # list of obsids or tile ids to combine
        cutout_cen=None,  # central coordinates of the cutout
        cutout_size=None,  # size of the cutout in angular units
        name=None,  # suffix for the output file basename.
        individual_dithers=False,  # if True, dithers are combined separately
        bkg_sub=True,  # to subtract background or not
        bkg_mesh_size=None,  # size of the background mesh boxes in angular units
        filter_size=3,  # median filter background over `filter_size` x `filter_size` boxes
        release_name="Q1_R1",  # the data release to use to find obs_ids if required
        instrument=None,  # Euclid instrument,e.g., VIS or NISP in nicl.euclid.constants
        swarp_config=None,  # SWarp configuration file content
        overwrite=False,  # overwrite existing combined image files
        debug=False,  # retain intermediate files for checking
        **kwargs,  # command line arguments to pass to SWarp, will override the defaults
    ):
        """
        Initialize the Combine class.

        Parameters:
            in_dir (str, optional): if not specified it will use the default data path based on release_name
            out_dir (str, optional): if not specified it will use the default test data output dir based on release_name
            filters (list, optional): if not specified it will default to all filters available for a given instrument
            ids (list, optional): this or cutout_cen and cutout_size must be specified. if obs_ids is not provided, they will be inferred from cutout_cen and cutout_size.
            cutout_cen (atsropy SkyCoord or str, optional): SkyCoord object or a string in the format "ra dec" (e.g. "00:00:00.0 +00:00:00.0")
            cutout_size (tuple, optional): single or a tuple of astropy Quantity object in angular units (e.g. (1*u.arcmin, 1*u.arcmin))
            name (str, optional): suffix for the output file basename. Default is a string of all obs_ids chained.
            bkg_sub (bool, optional): Whether to subtract background or not. Default is True.
            bkg_mesh_size (float, optional): single or a tuple of astropy Quantity object in angular units (e.g. (10*u.arcsec, 10*u.arcsec))
            filter_size (int, optional): Median filter background over `filter_size` x `filter_size` boxes. Default is 3.
            release_name (str, optional): The data release to use to find data if required. Default is "Q1_R1" for the current data release.
            instrument (Euclid Constant, optional): Euclid instrument, e.g., VIS or NISP in nicl.euclid.constants. Default is None.
            swarp_config (str, optional): SWarp configuration file content. Default is None.
            overwrite (bool, optional): Whether to overwrite existing combined image files. Default is False.
            debug (bool, optional): Whether to retain intermediate files for checking. Default is False.
        """
        # TODO: initialization is lengthy, should be refactored in the future
        self.instrument = instrument
        self.swarp_config = swarp_config
        self.bkg_sub = bkg_sub
        self.filter_size = filter_size
        self.release_name = release_name
        self.individual_dithers = individual_dithers
        self.overwrite = overwrite
        self.debug = debug
        if in_dir is None:
            if self.release_name is not None:
                self.in_path = default_data_path(release_name)
            else:
                raise ValueError(
                    "Either in_path or release_name must be specified to find the input images."
                )
        else:
            in_dir = Path(in_dir).expanduser()
            if in_dir.is_dir() and in_dir.exists():
                self.in_path = in_dir
            else:
                raise ValueError("The input path does not exist or is not a directory.")
        if cutout_cen is not None:
            if isinstance(cutout_cen, str):
                self.cutout_cen = SkyCoord(cutout_cen, unit=(u.hourangle, u.deg))
            elif isinstance(cutout_cen, SkyCoord):
                self.cutout_cen = cutout_cen
            else:
                raise ValueError(
                    "cutout_cen must be a string in the format 'ra dec' or an astropy.coordinates.SkyCoord object."
                )
        else:
            self.cutout_cen = None
        if cutout_size is not None:
            if isinstance(cutout_size, (int, float)):
                self.cutout_size = (cutout_size * u.arcsec, cutout_size * u.arcsec)
            if isinstance(cutout_size, u.Quantity) and cutout_size.unit.is_equivalent(
                u.arcsec
            ):
                self.cutout_size = (cutout_size, cutout_size)
            elif len(cutout_size) == 2:
                if isinstance(cutout_size[0], (int, float)) and isinstance(
                    cutout_size[1], (int, float)
                ):
                    self.cutout_size = (
                        cutout_size[0] * u.arcsec,
                        cutout_size[1] * u.arcsec,
                    )
                elif (
                    isinstance(cutout_size[0], u.Quantity)
                    and cutout_size[0].unit.is_equivalent(u.arcsec)
                    and isinstance(cutout_size[1], u.Quantity)
                    and cutout_size[1].unit.is_equivalent(u.arcsec)
                ):
                    self.cutout_size = cutout_size
                else:
                    raise ValueError(
                        "cutout_size is inhomogeneous. it should be either plain numbers or astropy.units.Quantity."
                    )
            else:
                raise ValueError(
                    "cutout_size must be a single quantity or a tuple of two quantities."
                )
        else:
            self.cutout_size = None
        if ids is None:
            # try getting ids from cutout_cen and cutout_size
            if self.cutout_cen is not None and self.cutout_size is not None:
                self.ids = self._get_ids()
                if not self.ids:
                    raise ValueError(
                        "No observations found for the given cutout parameters."
                    )
            else:
                raise ValueError(
                    "Either ids or cutout_cen and cutout_size must be specified."
                )
        else:
            if isinstance(ids, (int, str)):
                ids = [ids]
            self.ids = ids
        if name is None and len(ids) >= 1:
            self.name = "-".join([str(obs_id) for obs_id in ids])
        else:
            self.name = name
        if out_dir is None:
            if self.release_name is not None:
                self.out_dir = default_data_path(f"{self.release_name}_test", name)
            else:
                raise ValueError(
                    "Either out_dir or release_name must be specified to infer where to place the output images."
                )
        else:
            out_dir = Path(out_dir).expanduser()
            if out_dir.exists() and out_dir.is_file():
                raise ValueError("The output directory points to a file.")
            self.out_dir = out_dir
        if bkg_mesh_size is not None:
            if isinstance(bkg_mesh_size, (int, float)):
                self.bkg_mesh_size = (
                    bkg_mesh_size * u.arcsec,
                    bkg_mesh_size * u.arcsec,
                )
            if isinstance(
                bkg_mesh_size, u.Quantity
            ) and bkg_mesh_size.unit.is_equivalent(u.arcsec):
                self.bkg_mesh_size = (bkg_mesh_size, bkg_mesh_size)
            elif len(bkg_mesh_size) == 2:
                if isinstance(bkg_mesh_size[0], (int, float)) and isinstance(
                    bkg_mesh_size[1], (int, float)
                ):
                    self.bkg_mesh_size = (
                        bkg_mesh_size[0] * u.arcsec,
                        bkg_mesh_size[1] * u.arcsec,
                    )
                elif (
                    isinstance(bkg_mesh_size[0], u.Quantity)
                    and bkg_mesh_size[0].unit.is_equivalent(u.arcsec)
                    and isinstance(bkg_mesh_size[1], u.Quantity)
                    and bkg_mesh_size[1].unit.is_equivalent(u.arcsec)
                ):
                    self.bkg_mesh_size = bkg_mesh_size
                else:
                    raise ValueError(
                        "bkg_mesh_size is inhomogeneous. it should be either plain numbers or astropy.units.Quantity."
                    )
            else:
                raise ValueError(
                    "bkg_mesh_size must be a single quantity or a tuple of two quantities."
                )
        else:
            self.bkg_mesh_size = None
        if filters is None:
            if self.instrument is not None:
                self.filters = self.instrument.filters
            else:
                raise ValueError(
                    "Either filters or instrument must be specified to determine the filters."
                )
        else:
            filters = np.atleast_1d(filters)
            if self.instrument is not None:
                if set(filters).issubset(self.instrument.filters):
                    self.filters = filters
                else:
                    raise ValueError(
                        "The filters provided are not available for the specified instrument."
                    )
            else:
                self.filters = filters
        self._swarp_extra_args = self._parse_swarp_args(kwargs)

    def combine(self):
        for filter in np.atleast_1d(self.filters):
            self.combine_per_filter(filter)

    @abstractmethod
    def combine_per_filter(self, filter):
        pass

    def _parse_swarp_args(self, kwargs):
        kwargs = {k.upper(): v for k, v in kwargs.items()}
        swarp_args = []
        for key, value in kwargs.items():
            if key in ["CENTER", "IMAGE_SIZE"]:
                raise ValueError(
                    f"{key} should be supplied as input arguments to {self.__class__.__name__}"
                )
            swarp_args.extend([f"-{key}", str(value)])
        if self.cutout_size is not None:
            if "PIXEL_SCALE" in kwargs:
                pixel_scale = kwargs["PIXEL_SCALE"]
            else:
                pixel_scale = self.instrument.pix_scale
            cutout_size_pix = [
                round(cutsize.to(u.arcsec).value / pixel_scale)
                for cutsize in self.cutout_size
            ]
            swarp_args.extend(
                ["-IMAGE_SIZE", f"{cutout_size_pix[0]},{cutout_size_pix[1]}"]
            )
        if self.cutout_cen is not None:
            swarp_args.extend(
                [
                    "-CENTER",
                    f"{self.cutout_cen.to_string('hmsdms', sep=':', precision=2)}",
                    "-CENTER_TYPE",
                    "MANUAL",
                ]
            )
        return swarp_args

    @abstractmethod
    def _get_ids(self):
        pass

    @abstractmethod
    def _find_images(self):
        pass

    @abstractmethod
    def _prepare_input_for_swarp(self):
        pass

    def _run_swarp(self, tmpdir):
        print("Running SWarp...")
        with open(tmpdir / "config.swarp", "w") as f:
            f.write(self.swarp_config)
        swarp_cmd = ["swarp", "@images.list", "-c", "config.swarp"]
        swarp_cmd.extend(self._swarp_extra_args)
        try:
            run_swarp = subprocess.run(
                swarp_cmd,
                cwd=tmpdir,
                text=True,
                capture_output=True,
                check=True,
            )
            if self.debug:
                print(run_swarp.stderr)
            print("SWarp finished successfully.")
        except subprocess.CalledProcessError as e:
            print(
                f"Command '{' '.join(e.cmd)}' returned non-zero exit status, please check its stderr below."
            )
            print(e.stderr)

    @abstractmethod
    def _post_process(self):
        pass

In [ ]:
# | export
# mixin class for pre and post processing VIS and NISP dithers
# for reducing redundancy because VISCombiner and NISPCombiner are similar


class DithersMixin:
    def _get_obsids(self):
        da = DataAccess(release_name=self.release_name)
        radius = 0.5 * max(*self.cutout_size)
        ids = da.find_observations_for_target(
            self.cutout_cen.ra, self.cutout_cen.dec, radius, fully_contained=False
        )
        print(f"Determined required observation ids: {ids}")
        return ids

    def _find_dithers(self, pattern, filter):
        """find paths to dithers for a given filter"""
        image_paths = []
        for id in np.atleast_1d(self.ids):
            image_paths.extend(
                self.in_path.glob(pattern.format(obsid=id, filter=filter))
            )
        n_dithers = len(image_paths)
        n_obs = len(self.ids)
        if n_dithers == 0:
            print("Found no files.")
        elif n_dithers < self.instrument.n_dithers_per_obs * n_obs:
            print("Missing some files.")
        elif n_dithers > self.instrument.n_dithers_per_obs * n_obs:
            print("Found too many files.")
        return image_paths

    def _prepare_dithers_for_swarp(self, dithers, tmpdir):
        print("Preparing science and weight images for swarp...")
        sci_fns = []
        for i, dither in enumerate(dithers):
            with fits.open(dither) as hdul:
                for ext in self.instrument.extnames:
                    sci_data = hdul[f"{ext}.SCI"].data
                    rms_data = hdul[f"{ext}.RMS"].data
                    if self.instrument.name == "VIS":
                        dq_data = hdul[f"{ext}.FLG"].data
                    elif self.instrument.name == "NISP":
                        dq_data = hdul[f"{ext}.DQ"].data
                    primary_hdr = hdul[0].header
                    sci_ext_hdr = hdul[f"{ext}.SCI"].header
                    rms_ext_hdr = hdul[f"{ext}.RMS"].header
                    bad_pix_mask = np.any(
                        [
                            (dq_data & 2**bit > 0)
                            for bit in self.instrument.bad_pix_bits
                        ],
                        axis=0,
                    )
                    # subtract background if requested
                    if self.bkg_sub:
                        obj_mask, _ = fast_mask(sci_data, estimate_background=True)
                        # combine the bad pixel mask and the object mask to form a final mask
                        mask = bad_pix_mask | obj_mask
                        # 1 x 1 mesh by default if not specified
                        if self.bkg_mesh_size is None:
                            bkg_mesh_size_pix = (
                                sci_data.shape[0],
                                sci_data.shape[1],
                            )
                        else:
                            bkg_mesh_size_pix = (
                                round_up_box_size(
                                    sci_data.shape[0],
                                    self.bkg_mesh_size[0].to(u.arcsec).value
                                    / self.instrument.pix_scale,
                                ),
                                round_up_box_size(
                                    sci_data.shape[1],
                                    self.bkg_mesh_size[0].to(u.arcsec).value
                                    / self.instrument.pix_scale,
                                ),
                            )
                        bg = Background2D(
                            sci_data,
                            bkg_mesh_size_pix,
                            mask=mask,
                            filter_size=(self.filter_size, self.filter_size),
                            sigma_clip=SigmaClip(sigma=3),
                            exclude_percentile=90.0,
                        )
                        # subtract the background
                        sci_data -= bg.background
                        if self.instrument.name == "NISP":
                            # compute FLXSCALE for swarp only for NISP; save the value to PHOSCALE because FLXSCALE is occupied
                            # VIS already has the proper FLXSCALE in the headers
                            exptime = primary_hdr["EXPTIME"]
                            photfnu = primary_hdr["PHOTFNU"]
                            phrelex = primary_hdr["PHRELEX"]
                            phreldt = sci_ext_hdr["PHRELDT"]
                            phoscale = (1.0 / exptime) * photfnu * phrelex * phreldt
                            sci_ext_hdr.append(
                                (
                                    "PHOSCALE",
                                    phoscale,
                                    "Combined photometric scaling factors",
                                )
                            )
                    # compute weight map
                    with np.errstate(divide="ignore"):
                        np.divide(1.0, np.square(rms_data), out=rms_data)
                    # set the weight to 0 for bad pixels
                    rms_data[bad_pix_mask] = 0
                    sci_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(sci_data, sci_ext_hdr),
                        ]
                    )
                    weight_hdul = fits.HDUList(
                        [
                            fits.PrimaryHDU(header=primary_hdr),
                            fits.ImageHDU(rms_data, rms_ext_hdr),
                        ]
                    )
                    sci_fn = f"sci_{i}_{ext}.fits"
                    wt_fn = f"sci_{i}_{ext}.weight.fits"
                    sci_fns.append(sci_fn)
                    sci_hdul.writeto(tmpdir / sci_fn)
                    weight_hdul.writeto(tmpdir / wt_fn)
        # prepare image list for swarp
        with open(tmpdir / "images.list", "w") as f:
            for fn in sci_fns:
                f.write(f"{fn}[1]\n")

    def _post_process_stack_and_weight(self, tmpdir, out_fn):
        """clean up FITS headers and copy the output to the desired directory"""
        with fits.open(tmpdir / "coadd.fits", memmap=True) as hdul_sci, fits.open(
            tmpdir / "coadd.weight.fits", memmap=True
        ) as hdul_weights:
            # this is necessary because they are both PrimaryHDU objects
            hdu_sci = fits.ImageHDU(hdul_sci[0].data, hdul_sci[0].header)
            hdu_rms = fits.ImageHDU(hdul_weights[0].data, hdul_weights[0].header)
            # set science image to NaN where the weight is zero
            hdu_sci.data[hdu_rms.data == 0] = np.nan
            # convert the weight to RMS
            with np.errstate(divide="ignore"):
                np.divide(1.0, hdu_rms.data, out=hdu_rms.data)
            np.sqrt(hdu_rms.data, out=hdu_rms.data)
            # clean up the headers
            hdu_sci.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_sci.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_sci.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_sci.header.set("EXTNAME", "SCI", "Extension name", after="GCOUNT")
            hdu_sci.header.set("ZPAB", 23.9)
            hdu_sci.header.set("ZPABE", 0.0)
            hdu_sci.header.set("ZPVEGA", 1.0)
            hdu_sci.header.set("ZPVEGAE", 0.0)
            hdu_sci.header.remove("SIMPLE", ignore_missing=True)
            hdu_rms.header.set("XTENSION", "IMAGE   ", "Image extension", before=True)
            hdu_rms.header.set("PCOUNT", 0, "number of parameters", after="NAXIS2")
            hdu_rms.header.set("GCOUNT", 1, "number of groups", after="PCOUNT")
            hdu_rms.header.set("EXTNAME", "RMS", "Extension name", after="GCOUNT")
            for key in ("SIMPLE", "ZPAB", "ZPAB", "ZPVEGA", "ZPVEGAE"):
                hdu_rms.header.remove(key, ignore_missing=True)
            hdul = fits.HDUList([fits.PrimaryHDU(), hdu_sci, hdu_rms])
            self.out_dir.mkdir(parents=True, exist_ok=True)
            hdul.writeto(self.out_dir / out_fn, overwrite=self.overwrite)

In [ ]:
# | export
# subclass for combing NISP images


class NISPCombiner(DithersMixin, Combiner):
    """Combine NISP images using SWarp"""

    def __init__(self, **kwargs):
        super().__init__(instrument=NISP, swarp_config=SWARP_CONFIG_NISP, **kwargs)

    def combine_per_filter(self, filter):
        images = self._find_images(filter)
        if not images:
            return
        out_fn = "EUC_NIR_W-STK_" + filter
        if len(self.name) > 0:
            out_fn += f"_{self.name}"
        out_fn += ".fits"
        if self.individual_dithers:
            for image in images:
                dither = get_dither_id_from_filename(image)
                self._combine_images(
                    [image], out_fn.replace(".fits", f"_{dither}.fits")
                )
        else:
            self._combine_images(images, out_fn)

    def _combine_images(self, images, out_fn):
        with tempfile.TemporaryDirectory(delete=(not self.debug)) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            self._prepare_input_for_swarp(images, tmpdir)
            self._run_swarp(tmpdir)
            self._post_process(tmpdir, out_fn)

    def _get_ids(self):
        return super()._get_obsids()

    def _find_images(self, filter):
        """return a list of paths to the NISP dithers"""
        return super()._find_dithers(
            pattern="**/{obsid}/EUC_NIR_W-CAL-IMAGE_{filter}-{obsid}-*_*.fits",
            filter=filter,
        )

    def _prepare_input_for_swarp(self, images, tmpdir):
        super()._prepare_dithers_for_swarp(images, tmpdir)

    def _post_process(self, tmpdir, out_fn):
        super()._post_process_stack_and_weight(tmpdir, out_fn)

In [ ]:
# | export


class VISCombiner(DithersMixin, Combiner):
    """Combine VIS images using SWarp"""

    def __init__(self, **kwargs):
        super().__init__(instrument=VIS, swarp_config=SWARP_CONFIG_VIS, **kwargs)

    def combine_per_filter(self, filter):
        images = self._find_images()
        if not images:
            return
        out_fn = "EUC_VIS_SWL-STK_" + filter
        if len(self.name) > 0:
            out_fn += f"-{self.name}"
        out_fn += ".fits"
        if self.individual_dithers:
            for image in images:
                dither = get_dither_id_from_filename(image)
                self._combine_images(
                    [image], out_fn.replace(".fits", f"_{dither}.fits")
                )
        else:
            self._combine_images(images, out_fn)

    def _combine_images(self, images, out_fn):
        # the default temporary directory may not have enough space for VIS
        # determine the parent directory for the temporary directory based on the host
        hostname = socket.gethostname()
        username = getpass.getuser()
        match hostname:
            case "odhar":
                tmp_dir_parent = Path(f"/data/{username}")
            case "captain":
                tmp_dir_parent = Path("~").expanduser()
            case _:
                tmp_dir_parent = None
        with tempfile.TemporaryDirectory(
            dir=tmp_dir_parent, delete=(not self.debug)
        ) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            self._prepare_input_for_swarp(images, tmpdir)
            self._run_swarp(tmpdir)
            self._post_process(tmpdir, out_fn)

    def _find_images(self):
        """return a list of paths to the VIS dithers"""
        return super()._find_dithers(
            pattern="**/{obsid}/EUC_VIS_SWL-DET-*{obsid}-*-*_*.fits",
            filter="I",
        )

    def _prepare_input_for_swarp(self, images, tmpdir):
        super()._prepare_dithers_for_swarp(images, tmpdir)

    def _post_process(self, tmpdir, out_fn):
        super()._post_process_stack_and_weight(tmpdir, out_fn)

In [ ]:
# | export


class MerCombiner(Combiner):
    """Combine MER stacks using SWarp"""

    def __init__(self, **kwargs):
        if "add_bkg_mod" in kwargs:
            self.add_bkg_mod = kwargs["add_bkg_mod"]
            # remove it from kwargs
            kwargs.pop("add_bkg_mod")
        super().__init__(
            instrument=MER, swarp_config=SWARP_CONFIG_MER, bkg_sub=False, **kwargs
        )

    def combine_per_filter(self, filter):
        images = self._find_images(filter)
        if not list(chain.from_iterable(images)):
            return
        out_fn = (
            "EUC_MER-MOSAIC-" if self.add_bkg_mod else "EUC_MER_BGSUB-MOSAIC-"
        ) + filter
        if len(self.name) > 0:
            out_fn += f"_{self.name}"
        out_fn += ".fits"
        with tempfile.TemporaryDirectory(delete=(not self.debug)) as tmpdir:
            tmpdir = Path(tmpdir)
            if self.debug:
                print(f"Intermediate files can be found in {tmpdir}/.")
                print("You must delete this folder manually when done.")
            self._prepare_input_for_swarp(images, tmpdir)
            self._run_swarp(tmpdir)
            self._post_process(tmpdir, out_fn)

    def _get_ids(self):
        da = DataAccess(release_name=self.release_name)
        radius = 0.5 * max(*self.cutout_size)
        ids = da.find_tiles_for_target(
            self.cutout_cen.ra, self.cutout_cen.dec, radius, fully_contained=False
        )
        print(f"Determined required tile ids: {ids}")
        return ids

    def _find_images(self, filter):
        """return a list of tuple of paths to the MER stacks and bkg models"""
        images = []
        for tile_id in np.atleast_1d(self.ids):
            img_tpl = []
            img_tpl.extend(
                self.in_path.glob(
                    f"**/EUC_MER_BGSUB-MOSAIC-{filter}_TILE{tile_id}-*.fits"
                ),
            )
            if self.add_bkg_mod:
                img_tpl.extend(
                    self.in_path.glob(
                        f"**/EUC_MER_BGMOD*-{filter}_TILE{tile_id}-*.fits"
                    )
                )
            else:
                img_tpl.append(None)
            images.append(tuple(img_tpl))
        n_images = sum(1 for img in chain.from_iterable(images) if img is not None)
        n_tiles = len(self.ids)
        n_images_per_tile = 2 if self.add_bkg_mod else 1
        if n_images == 0:
            print("Found no files.")
        elif n_images < n_tiles * n_images_per_tile:
            print("Missing some files.")
        elif n_images > n_tiles * n_images_per_tile:
            print("Found too many files.")
        return images

    def _prepare_input_for_swarp(self, images, tmpdir):
        print("Preparing MER stacks for swarp...")
        sci_fns = []
        for stack, bkg_mod in images:
            with fits.open(stack) as hdul:
                data = hdul[0].data
                hdr = hdul[0].header
                # I believe that all MER stacks of a given filter have the same ZP
                # So the codes below is just for safeguard purpose
                filter = hdr.get("FILTER", None)
                mag_zp = hdr.get("MAGZERO", None)
                if filter is None or mag_zp is None:
                    corr_factor = 1
                else:
                    match filter:
                        case "VIS":
                            expected_mag_zp = 24.6
                        case "NIR_Y":
                            expected_mag_zp = 29.8
                        case "NIR_J":
                            expected_mag_zp = 30.0
                        case "NIR_H":
                            expected_mag_zp = 29.9
                        case _:
                            expected_mag_zp = None
                    if mag_zp != expected_mag_zp:
                        corr_factor = 10 ** (0.4 * (expected_mag_zp - mag_zp))
                        hdr.set("MAGZERO", expected_mag_zp)
                    else:
                        corr_factor = 1
                # add bkg model back to the image if requested
                if self.add_bkg_mod:
                    with fits.open(bkg_mod) as hdul_bkg:
                        bkg_mod = hdul_bkg[0].data
                        data += bkg_mod
                # determine if writing temp fits files is needed
                if corr_factor != 1 or self.add_bkg_mod:
                    data *= corr_factor
                    sci_hdul = fits.HDUList(fits.PrimaryHDU(data=data, header=hdr))
                    # preserve the original file name for debugging
                    sci_fns.append(stack.name)
                    sci_hdul.writeto(tmpdir / stack.name)
                else:
                    # no bkg mod to add and no ZP correction needed, simply use the original file
                    sci_fns.append(str(stack))
        # write image list file for swarp
        with open(tmpdir / "images.list", "w") as f:
            for fn in sci_fns:
                f.write(f"{fn}\n")

    def _post_process(self, tmpdir, out_fn):
        """copy the final stack to the desired directory"""
        self.out_dir.mkdir(parents=True, exist_ok=True)
        if not self.overwrite and (self.out_dir / out_fn).exists():
            raise FileExistsError(
                f"Output stack file {self.out_dir / out_fn} already exists."
            )
        copy2(tmpdir / "coadd.fits", self.out_dir / out_fn)

## Examples

Produce a H-band stack for a single Q1 NISP observation.

In [ ]:
in_dir = default_data_path("Q1_R1")
id = 2682
out_dir = default_data_path("Q1_R1_processed_test", f"stacked/{id}")

In [ ]:
%%time
combiner = NISPCombiner(in_dir=in_dir, out_dir=out_dir, ids=id, filters="H", overwrite=True)
combiner.combine()

Preparing science and weight images for swarp...
Running SWarp...
SWarp finished successfully.
CPU times: user 2min 10s, sys: 17.5 s, total: 2min 27s
Wall time: 3min 57s


Produce a H-band stacks of individual dithers for a single Q1 observation.

In [ ]:
%%time
combiner = NISPCombiner(
    in_dir=in_dir,
    out_dir=out_dir,
    ids=id,
    filters="H",
    individual_dithers=True,
    overwrite=True,
)
combiner.combine()

Preparing science and weight images for swarp...
Running SWarp...
SWarp finished successfully.
Preparing science and weight images for swarp...
Running SWarp...
SWarp finished successfully.
Preparing science and weight images for swarp...
Running SWarp...
SWarp finished successfully.
Preparing science and weight images for swarp...
Running SWarp...
SWarp finished successfully.
CPU times: user 2min 13s, sys: 19.7 s, total: 2min 33s
Wall time: 4min 4s


Produce a H-band stack image of a galaxy cluster from Euclid Q1 data.

First, with an angular size that is contained within a single observation.

In [ ]:
cluster_id = "MCXCJ1754.6+6803"
z = 0.07
ra = 268.662 * u.deg
dec = 68.058 * u.deg
cutout_radius_physical = 500 * u.kpc
cluster_path = default_data_path("Q1_R1_clusters_test", cluster_id)
cutout_radius_angular = physical_to_angular(cutout_radius_physical, z)
cutout_size = 2 * cutout_radius_angular
bkg_mesh_size = 0.5 * cutout_radius_angular

In [ ]:
# here we use original data from the archive an save to a test location
in_dir = default_data_path("Q1_R1")
out_dir = default_data_path("Q1_R1_clusters_test", cluster_id)

# to use processed data of a particular version and processing stage
# in_path = default_data_path("Q1_R1_processed_v0.2", "persistence")
# out_path = default_data_path("Q1_R1_clusters_v0.2", cluster_id)

In [ ]:
combiner = NISPCombiner(
    in_dir=in_dir,
    out_dir=out_dir,
    bkg_mesh_size=bkg_mesh_size,
    cutout_cen=SkyCoord(ra, dec),
    cutout_size=cutout_size,
    name=cluster_id,
    filters="H",
    overwrite=True,
)
combiner.combine()

INFO: OK [astroquery.utils.tap.core]
Determined required observation ids: [2683]
Preparing science and weight images for swarp...
Running SWarp...
SWarp finished successfully.


Second, with an angular size that requires multiple observations (2682, 2683, 2685) to be combined.

In [ ]:
cutout_size = 6 * cutout_radius_angular
bkg_mesh_size = 0.5 * cutout_radius_angular
name = f"{cluster_id}-big"

In [ ]:
combiner = NISPCombiner(
    in_dir=in_dir,
    out_dir=out_dir,
    bkg_mesh_size=bkg_mesh_size,
    cutout_cen=SkyCoord(ra, dec),
    cutout_size=cutout_size,
    name=name,
    filters="H",
    overwrite=True,
)
combiner.combine()

INFO: OK [astroquery.utils.tap.core]
Determined required observation ids: [2682 2683 2685]
Preparing science and weight images for swarp...
Running SWarp...
SWarp finished successfully.


In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()